In [33]:
import json
import numpy as np
import pandas as pd

In [34]:
import pandas as pd


# Read the Excel file into a DataFrame
file_path = "D:\OVGU\AMS project\GIT\AMS_ShipTraj\physics based model\Rheinhafen Data.xlsx"
df = pd.read_excel(file_path, header=None)
print(df.head())

         0                       1  \
0  3730963 2025-03-09 00:00:00.397   
1  3730964 2025-03-09 00:00:04.230   
2  3730965 2025-03-09 00:00:06.277   
3  3730966 2025-03-09 00:00:07.531   
4  3730967 2025-03-09 00:00:10.537   

                                                   2  
0  {"msg_type": 1, "repeat": 0, "mmsi": 244660361...  
1  {"msg_type": 1, "repeat": 0, "mmsi": 211576360...  
2  {"msg_type": 1, "repeat": 0, "mmsi": 211478290...  
3  {"msg_type": 1, "repeat": 0, "mmsi": 211632780...  
4  {"msg_type": 1, "repeat": 0, "mmsi": 244660361...  


In [35]:
time_series = df[1]
json_series = df[2]

records = []

for time_val, json_str in zip(time_series, json_series):
    try:
        # Parse the valid JSON string directly
        data_dict = json.loads(json_str)
        data_dict["time"] = str(time_val)
        records.append(data_dict)
    except Exception as e:
        print(f"Error processing record at time {time_val}: {e}")

# Optional: print first record
print(records[0])

{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.262617, 'lat': 48.981312, 'course': 46.9, 'heading': 47, 'second': 57, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2240, 'time': '2025-03-09 00:00:00.397000'}


In [36]:
for i in range(5):
    print(records[i])

{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.262617, 'lat': 48.981312, 'course': 46.9, 'heading': 47, 'second': 57, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2240, 'time': '2025-03-09 00:00:00.397000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 211576360, 'status': 0, 'turn': 0.0, 'speed': 0.0, 'accuracy': True, 'lon': 8.343733, 'lat': 49.015103, 'course': 356.0, 'heading': 180, 'second': 9, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2236, 'time': '2025-03-09 00:00:04.230000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 211478290, 'status': 0, 'turn': 0.0, 'speed': 0.0, 'accuracy': True, 'lon': 8.307823, 'lat': 49.016417, 'course': 238.2, 'heading': 358, 'second': 5, 'maneuver': 0, 'spare_1': 'AA==', 'raim': False, 'radio': 98571, 'time': '2025-03-09 00:00:06.277000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 211632780, 'status': 15, 'turn': -128.0, 'speed': 0.0, 'accuracy': True, 'lon': 8.25518, 'lat': 

In [37]:
from datetime import datetime

# Step 1: Filter by MMSI
target_mmsi = 244660361
filtered_records = [record for record in records if record['mmsi'] == target_mmsi]

# Step 2: Sort by time (convert 'time' to datetime during sorting)
sorted_records = sorted(
    filtered_records,
    key=lambda x: datetime.fromisoformat(x['time'])  # Converts '2025-03-09 00:00:00.397000' to datetime
)

# Optional: Show result
for rec in sorted_records[:3]:
    print(rec)

{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.262617, 'lat': 48.981312, 'course': 46.9, 'heading': 47, 'second': 57, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2240, 'time': '2025-03-09 00:00:00.397000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.263082, 'lat': 48.981605, 'course': 46.1, 'heading': 46, 'second': 7, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2234, 'time': '2025-03-09 00:00:10.537000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.1, 'accuracy': True, 'lon': 8.263542, 'lat': 48.981902, 'course': 45.9, 'heading': 46, 'second': 17, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 16384, 'time': '2025-03-09 00:00:20.351000'}


In [38]:
print(sorted_records[0]['lat'], sorted_records[0]['lon'], sorted_records[-1]['lat'], sorted_records[-1]['lon'])

48.981312 8.262617 49.098138 8.359107


In [39]:
import folium

# Ensure filtered_records is sorted by time
from datetime import datetime
filtered_records = sorted(
    filtered_records,
    key=lambda x: datetime.fromisoformat(x['time'])
)

# Find first valid lat/lon to center the map
for rec in filtered_records:
    if 'lat' in rec and 'lon' in rec:
        start_lat = rec['lat']
        start_lon = rec['lon']
        break

print(start_lat, start_lon)

# Initialize the map
m = folium.Map(location=[start_lat, start_lon], zoom_start=13)

# Add small circle markers for each valid point
for rec in filtered_records:
    lat = rec.get('lat')
    lon = rec.get('lon')
    time = rec.get('time')

    if lat is not None and lon is not None:
        folium.CircleMarker(
            location=[lat, lon],
            radius=0.1,  # 🔻 Smaller point size
            popup=f"Time: {time}",
            color='blue',
            fill=True,
            fill_opacity=0.6
        ).add_to(m)

# Add red pin at the start and green pin at the end
valid_path = [rec for rec in filtered_records if 'lat' in rec and 'lon' in rec]

if valid_path:
    # Red pin for start
    folium.Marker(
        location=[valid_path[0]['lat'], valid_path[0]['lon']],
        popup="Start",
        icon=folium.Icon(color='red', icon='play')
    ).add_to(m)

    # Green pin for end
    folium.Marker(
        location=[valid_path[-1]['lat'], valid_path[-1]['lon']],
        popup="End",
        icon=folium.Icon(color='green', icon='flag')
    ).add_to(m)

    # Optional: Draw path
    path = [(rec['lat'], rec['lon']) for rec in valid_path]
    folium.PolyLine(path, color='blue', weight=2.5, opacity=0.7).add_to(m)

# Save to file
m.save("mmsi_track_map.html")
# Display the map in Jupyter Notebook (if applicable)
m

48.981312 8.262617


48.97966, 8.25874

49.10166, 8.36027

In [40]:
import math
import folium

# Custom point
custom_point = {
    'lat': 48.97966,
    'lon': 8.25874,
    'course': 47  # not used now, since we're connecting directly
}

# Get the first valid point from the filtered records
for rec in filtered_records:
    if 'lat' in rec and 'lon' in rec:
        first_point = rec
        break

# --- Plotting ---

# Add the custom point marker
folium.Marker(
    location=[custom_point['lat'], custom_point['lon']],
    popup="Custom Point",
    icon=folium.Icon(color='orange', icon='plus')
).add_to(m)

# Draw a solid green line connecting the custom point directly to the first ship point
folium.PolyLine(
    locations=[
        (custom_point['lat'], custom_point['lon']),
        (first_point['lat'], first_point['lon'])
    ],
    color='green',
    weight=3,
    tooltip='Direct connection to ship path'
).add_to(m)

# Save the final map
m.save("mmsi_track_map_with_direct_connection.html")
# Display the final map in Jupyter Notebook (if applicable)
m

In [41]:
# Custom end point
end_point = {
    'lat': 49.10166,
    'lon': 8.36027,
    'course' : 47  
}

# Find the last valid point from the ship trajectory
for rec in reversed(filtered_records):
    if 'lat' in rec and 'lon' in rec:
        last_point = rec
        break

# Add a marker for the custom end point
folium.Marker(
    location=[end_point['lat'], end_point['lon']],
    popup="End Point",
    icon=folium.Icon(color='darkgreen', icon='stop')
).add_to(m)

# Draw a solid green line from ship's last point to the custom end point
folium.PolyLine(
    locations=[
        (last_point['lat'], last_point['lon']),
        (end_point['lat'], end_point['lon'])
    ],
    color='green',
    weight=3,
    tooltip='Connection to end point'
).add_to(m)

# Save updated map
m.save("mmsi_track_map_with_start_and_end_connection.html")
# Display the updated map in Jupyter Notebook (if applicable)
m

In [42]:
"""# %%
import numpy as np
from scipy.interpolate import CubicHermiteSpline
import folium

# Define all 4 points in the required format
custom_start_point = {
    'lat': 48.97966,
    'lon': 8.25874,
    'course': 47
}

ais_start_point = {
    'lat': first_point['lat'],
    'lon': first_point['lon'],
    'course': first_point.get('course', 0)
}

ais_end_point = {
    'lat': last_point['lat'],
    'lon': last_point['lon'],
    'course': last_point.get('course', 0)
}

custom_end_point = {
    'lat': 49.10166,
    'lon': 8.36027,
    'course': 47
}

# Helper to convert course (deg) to dy/dx
def course_to_derivative(course_deg):
    theta = np.radians(course_deg)
    dx = np.cos(theta)
    dy = np.sin(theta)
    if abs(dx) < 1e-6:
        return np.sign(dy) * 1e6
    return dy / dx

# === 1. Custom Start → AIS Start ===
x_vals1 = [custom_start_point['lon'], ais_start_point['lon']]
y_vals1 = [custom_start_point['lat'], ais_start_point['lat']]
dy_dx1 = [
    course_to_derivative(custom_start_point['course']),
    course_to_derivative(ais_start_point['course'])
]

spline1 = CubicHermiteSpline(x_vals1, y_vals1, dy_dx1)
x_interp1 = np.linspace(x_vals1[0], x_vals1[1], 50)
y_interp1 = spline1(x_interp1)

# === 2. AIS End → Custom End ===
x_vals2 = [ais_end_point['lon'], custom_end_point['lon']]
y_vals2 = [ais_end_point['lat'], custom_end_point['lat']]
dy_dx2 = [
    course_to_derivative(ais_end_point['course']),
    course_to_derivative(custom_end_point['course'])
]

spline2 = CubicHermiteSpline(x_vals2, y_vals2, dy_dx2)
x_interp2 = np.linspace(x_vals2[0], x_vals2[1], 50)
y_interp2 = spline2(x_interp2)

# === Plot on folium map ===
folium.PolyLine(
    locations=list(zip(y_interp1, x_interp1)),
    color='purple',
    weight=3,
    tooltip='Cubic Spline: custom → AIS start'
).add_to(m)

folium.PolyLine(
    locations=list(zip(y_interp2, x_interp2)),
    color='orange',
    weight=3,
    tooltip='Cubic Spline: AIS end → custom'
).add_to(m)

# Save and display
m.save("mmsi_track_map_with_cubic_splines_course_based.html")
m
"""

'# %%\nimport numpy as np\nfrom scipy.interpolate import CubicHermiteSpline\nimport folium\n\n# Define all 4 points in the required format\ncustom_start_point = {\n    \'lat\': 48.97966,\n    \'lon\': 8.25874,\n    \'course\': 47\n}\n\nais_start_point = {\n    \'lat\': first_point[\'lat\'],\n    \'lon\': first_point[\'lon\'],\n    \'course\': first_point.get(\'course\', 0)\n}\n\nais_end_point = {\n    \'lat\': last_point[\'lat\'],\n    \'lon\': last_point[\'lon\'],\n    \'course\': last_point.get(\'course\', 0)\n}\n\ncustom_end_point = {\n    \'lat\': 49.10166,\n    \'lon\': 8.36027,\n    \'course\': 47\n}\n\n# Helper to convert course (deg) to dy/dx\ndef course_to_derivative(course_deg):\n    theta = np.radians(course_deg)\n    dx = np.cos(theta)\n    dy = np.sin(theta)\n    if abs(dx) < 1e-6:\n        return np.sign(dy) * 1e6\n    return dy / dx\n\n# === 1. Custom Start → AIS Start ===\nx_vals1 = [custom_start_point[\'lon\'], ais_start_point[\'lon\']]\ny_vals1 = [custom_start_point[\

In [43]:
# %%
from scipy.interpolate import CubicSpline

# Build the full sequence of lat-lon points
lat_lon_path = [
    (custom_start_point['lat'], custom_start_point['lon'])
] + [
    (rec['lat'], rec['lon']) for rec in filtered_records if 'lat' in rec and 'lon' in rec
] + [
    (custom_end_point['lat'], custom_end_point['lon'])
]

lat_lon_path = np.array(lat_lon_path)
lats = lat_lon_path[:, 0]
lons = lat_lon_path[:, 1]

# Parameter t for interpolation
t_vals = np.linspace(0, 1, len(lat_lon_path))

# Fit cubic spline
lat_spline = CubicSpline(t_vals, lats)
lon_spline = CubicSpline(t_vals, lons)

# Interpolate 300 smooth points
t_interp = np.linspace(0, 1, 300)
lat_interp = lat_spline(t_interp)
lon_interp = lon_spline(t_interp)

# Plot full cubic spline path
folium.PolyLine(
    locations=list(zip(lat_interp, lon_interp)),
    color='black',
    weight=3,
    tooltip='Full Cubic Spline: custom → AIS → custom'
).add_to(m)

# Save and view
m.save("mmsi_full_custom_spline_path.html")
m


In [44]:
# %%
from scipy.interpolate import CubicSpline
import numpy as np
import folium
import os

# Ensure output folder exists
output_dir = "spline_skip_maps"
os.makedirs(output_dir, exist_ok=True)

# Generate 16 folium maps with increasing AIS point skip
folium_maps = []

for skip in range(1, 17):
    # Skip AIS points by step
    reduced_records = filtered_records[::skip + 1]

    # Build lat/lon path
    lat_lon_path = [  # start
        (custom_start_point['lat'], custom_start_point['lon'])
    ] + [
        (rec['lat'], rec['lon']) for rec in reduced_records if 'lat' in rec and 'lon' in rec
    ] + [  # end
        (custom_end_point['lat'], custom_end_point['lon'])
    ]

    # Convert to array
    lat_lon_path = np.array(lat_lon_path)
    lats = lat_lon_path[:, 0]
    lons = lat_lon_path[:, 1]

    # Parameter t
    t_vals = np.linspace(0, 1, len(lat_lon_path))

    # Fit spline
    lat_spline = CubicSpline(t_vals, lats)
    lon_spline = CubicSpline(t_vals, lons)

    # Interpolate smooth path
    t_interp = np.linspace(0, 1, 300)
    lat_interp = lat_spline(t_interp)
    lon_interp = lon_spline(t_interp)

    # Find center to initialize folium map
    center_lat = lat_interp[0]
    center_lon = lon_interp[0]

    # Create new map
    map_k = folium.Map(location=[center_lat, center_lon], zoom_start=13)

    # Add spline path
    folium.PolyLine(
        locations=list(zip(lat_interp, lon_interp)),
        color='black',
        weight=3,
        tooltip=f'Cubic Spline with skip={skip}'
    ).add_to(map_k)

    # Add start and end markers
    folium.Marker(
        location=[custom_start_point['lat'], custom_start_point['lon']],
        popup='Start',
        icon=folium.Icon(color='red', icon='play')
    ).add_to(map_k)

    folium.Marker(
        location=[custom_end_point['lat'], custom_end_point['lon']],
        popup='End',
        icon=folium.Icon(color='green', icon='flag')
    ).add_to(map_k)

    # Save map
    filename = f"spline_skip_{skip}.html"
    filepath = os.path.join(output_dir, filename)
    map_k.save(filepath)

    # Store for reference
    folium_maps.append((skip, map_k, filepath))

print("✅ All 16 folium maps saved in:", output_dir)


✅ All 16 folium maps saved in: spline_skip_maps


In [45]:
# %%
# Configuration
output_dir = "spline_skip_maps"
output_grid_html = "spline_grid.html"

# HTML header and styling for 4x4 grid
html_header = """
<!DOCTYPE html>
<html>
<head>
    <title>Cubic Spline Skip Comparison</title>
    <style>
        body {
            font-family: Arial, sans-serif;
        }
        .grid-container {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            grid-gap: 10px;
            padding: 10px;
        }
        .grid-item {
            border: 1px solid #ccc;
            padding: 5px;
        }
        iframe {
            width: 100%;
            height: 300px;
            border: none;
        }
        h2 {
            text-align: center;
        }
    </style>
</head>
<body>
    <h2>Cubic Spline Paths with Varying AIS Skip Levels (1 to 16)</h2>
    <div class="grid-container">
"""

html_footer = """
    </div>
</body>
</html>
"""

# Construct iframe blocks
iframe_blocks = ""
for skip in range(1, 17):
    map_filename = f"spline_skip_maps/spline_skip_{skip}.html"
    iframe_blocks += f"""
    <div class="grid-item">
        <strong>Skip = {skip}</strong>
        <iframe src="{map_filename}"></iframe>
    </div>
    """

# Write to final HTML
with open(output_grid_html, 'w') as f:
    f.write(html_header + iframe_blocks + html_footer)

print(f"✅ 4×4 grid HTML saved as: {output_grid_html}")


✅ 4×4 grid HTML saved as: spline_grid.html


In [1]:
import geopandas as gpd
import folium

# Load GeoJSON file
input_path = "json for 48.981312 8.262617 49.098138 8.359107.geojson"  # Replace with your actual path
gdf = gpd.read_file(input_path)

# Keep only LineStrings
gdf = gdf[gdf.geometry.type == 'LineString']

# Find longest LineString
gdf['length'] = gdf.length
longest_line = gdf.loc[gdf['length'].idxmax()]
longest_geom = longest_line.geometry

# Create a Folium map centered at the midpoint
midpoint = longest_geom.interpolate(0.5, normalized=True)
m = folium.Map(location=[midpoint.y, midpoint.x], zoom_start=13)

# Add the longest path
folium.PolyLine(locations=[(pt[1], pt[0]) for pt in longest_geom.coords], color="blue", weight=4).add_to(m)
folium.Marker(location=(longest_geom.coords[0][1], longest_geom.coords[0][0]), popup="Start", icon=folium.Icon(color='green')).add_to(m)
folium.Marker(location=(longest_geom.coords[-1][1], longest_geom.coords[-1][0]), popup="End", icon=folium.Icon(color='red')).add_to(m)

# Save and show
m.save("longest_path_map.html")
print("Map saved as longest_path_map.html")


Map saved as longest_path_map.html


C:\Users\nisha\AppData\Local\Temp\ipykernel_127784\12228278.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['length'] = gdf.length
